In [1]:
import json
import logging
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple, Any
from sqlalchemy import create_engine

In [2]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('af_administration_mapping.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [3]:
class Configuration:
    """Centralized configuration for paths and settings."""

    def __init__(self, base_path: str = "."):
        self.base_path = Path(base_path)

        # Input paths
        self.flow_cascades_path = (
            self.base_path / "output" / "flow_cascades"
        )
        self.flow_adms_path = self.base_path / "output" / "flow_sqlite"
        self.flow_data_path = (
            self.base_path / ".." / ".." / "storage" / "akvo-flow" /
            "raw"
        )

        # Database paths
        self.mis_adm_sqlite = (
            self.base_path / ".." / ".." / "backend" / "source" /
            "administrator.sqlite"
        )

        # Output paths
        self.output_mapping_csv = (
            self.base_path / ".." / ".." / "storage" / "akvo-flow" /
            "akvo-flow" / "administration_mapping.csv"
        )
        self.output_missing_csv = (
            self.base_path / ".." / ".." / "storage" / "akvo-flow" /
            "akvo-flow" / "administration_missing.csv"
        )

        # Dataframe column definitions
        self.flow_adms_columns = [
            "flow_datapoint_id",
            "flow_form_id",
            "flow_question_id",
            "flow_value",
            "mis_value",
        ]

        self.flow_missing_adms_columns = [
            "id",
            "code",
            "name",
            "full_path",
        ]

In [4]:
class AdministrationMapper:
    """Handles mapping between Flow administration values and MIS IDs."""

    def __init__(self, config: Configuration):
        self.config = config
        self.mis_adm_lookup: Dict[Tuple[str, int], int] = {}
        self.mis_adm_names_by_level: Dict[
            int, List[Dict[str, Any]]
        ] = {}
        self.flow_adm_lookups: Dict[
            str, Dict[Tuple[str, Optional[str]], Dict[str, Any]]
        ] = {}

    def validate_paths(self) -> None:
        """Validate that all required paths exist."""
        if not self.config.mis_adm_sqlite.exists():
            raise FileNotFoundError(
                f"{self.config.mis_adm_sqlite} not found. "
                "Please run `python manage.py generate_sqlite` "
                "in the backend container to create the database."
            )

        if not self.config.flow_cascades_path.exists():
            raise FileNotFoundError(
                f"Flow cascades path not found: "
                f"{self.config.flow_cascades_path}"
            )

        if not self.config.flow_data_path.exists():
            raise FileNotFoundError(
                f"Flow data path not found: {self.config.flow_data_path}"
            )

    def load_flow_cascades(self) -> pd.DataFrame:
        """Load and combine all cascade CSV files into a single DataFrame."""
        logger.info(
            f"Loading cascade files from "
            f"{self.config.flow_cascades_path}"
        )

        csv_files = list(self.config.flow_cascades_path.glob("*.csv"))

        if not csv_files:
            logger.warning(
                f"No CSV files found in {self.config.flow_cascades_path}"
            )
            return pd.DataFrame()

        logger.info(f"Found {len(csv_files)} cascade CSV files")

        dfs = []
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                dfs.append(df)
                logger.debug(f"Loaded {csv_file.name}: {len(df)} rows")
            except Exception as e:
                logger.error(f"Failed to read {csv_file.name}: {e}")

        if not dfs:
            return pd.DataFrame()

        combined_df = pd.concat(dfs, ignore_index=True)
        logger.info(f"Combined cascade data: {len(combined_df)} rows")

        return combined_df

    def load_mis_administration_data(self) -> None:
        """Pre-load MIS administration data into a lookup dictionary."""
        logger.info("Pre-loading MIS administration data...")

        try:
            engine = create_engine(f"sqlite:///{self.config.mis_adm_sqlite}")

            query = """
            SELECT id, LOWER(TRIM(name)) as name_lower, level
            FROM nodes
            """

            mis_adm_data = pd.read_sql_query(query, engine)

            # Create lookup: (name_lower, level) -> id
            self.mis_adm_lookup = {
                (row['name_lower'], row['level']): row['id']
                for _, row in mis_adm_data.iterrows()
            }

            # Also create a list of names for substring matching
            # This handles hierarchical names like "Nadroga/Navosa"
            # matching "Nadroga"
            self.mis_adm_names_by_level = {}
            for _, row in mis_adm_data.iterrows():
                level = row['level']
                if level not in self.mis_adm_names_by_level:
                    self.mis_adm_names_by_level[level] = []
                self.mis_adm_names_by_level[level].append({
                    'name_lower': row['name_lower'],
                    'id': row['id']
                })

            logger.info(
                f"Loaded {len(self.mis_adm_lookup)} "
                f"MIS administration entries"
            )

        except Exception as e:
            logger.error(f"Failed to load MIS administration data: {e}")
            raise

    def load_flow_administration_data(
        self, sqlite_file: str
    ) -> Dict[Tuple[str, Optional[str]], Dict[str, Any]]:
        """Pre-load flow administration data for a specific SQLite file."""
        flow_sqlite_path = self.config.flow_adms_path / sqlite_file

        if not flow_sqlite_path.exists():
            logger.warning(f"Flow SQLite file not found: {flow_sqlite_path}")
            return {}

        # Return cached lookup if already loaded
        if sqlite_file in self.flow_adm_lookups:
            return self.flow_adm_lookups[sqlite_file]

        logger.info(
            f"Pre-loading flow administration data from {sqlite_file}..."
        )

        try:
            engine = create_engine(f"sqlite:///{flow_sqlite_path}")

            query = """
            SELECT n1.id, n1.code, n1.name, n2.name as parent_name,
                   LOWER(TRIM(n1.name)) as name_lower
            FROM nodes n1
            LEFT JOIN nodes n2 ON n1.parent = n2.id
            """

            flow_adm_data = pd.read_sql_query(query, engine)

            # Create lookup: (name_lower, parent_name) -> row data
            lookup = {}
            for _, row in flow_adm_data.iterrows():
                key = (row['name_lower'], row['parent_name'])
                lookup[key] = {
                    'id': row['id'],
                    'code': row['code'],
                    'name': row['name']
                }

            self.flow_adm_lookups[sqlite_file] = lookup
            logger.info(
                f"Loaded {len(lookup)} flow administration entries "
                f"from {sqlite_file}"
            )

            return lookup

        except Exception as e:
            logger.error(
                f"Failed to load flow administration data from "
                f"{sqlite_file}: {e}"
            )
            return {}

    def find_mis_administration_match(
        self, adm_value: str, adm_level: int
    ) -> Optional[int]:
        """
        Find a matching MIS administration ID using exact or substring match.

        Args:
            adm_value: The administration name to match (lowercase, trimmed)
            adm_level: The administration level

        Returns:
            The matching administration ID, or None if not found
        """
        # First try exact match
        lookup_key = (adm_value, adm_level)
        if lookup_key in self.mis_adm_lookup:
            return int(self.mis_adm_lookup[lookup_key])

        # If no exact match, try substring matching for hierarchical names
        # This handles cases like "Nadroga" matching "Nadroga/Navosa"
        if adm_level in self.mis_adm_names_by_level:
            for entry in self.mis_adm_names_by_level[adm_level]:
                db_name = entry['name_lower']
                # Check if adm_value is a substring of db_name
                # or if db_name is a substring of adm_value
                if adm_value in db_name or db_name in adm_value:
                    logger.debug(
                        f"Substring match: '{adm_value}' matched to "
                        f"'{db_name}' (ID: {entry['id']})"
                    )
                    return entry['id']

        return None

    def parse_cascade_answer(
        self, answer_value: str
    ) -> Tuple[str, Optional[int]]:
        """
        Parse a cascade answer value and extract flow_value and mis_value.

        Args:
            answer_value: The raw answer value (JSON string or plain text)

        Returns:
            Tuple of (flow_value, mis_value) where mis_value is None if not
            found
        """
        flow_value = answer_value
        mis_value = None

        try:
            json_parsed = json.loads(answer_value)

            # Extract all names and join with pipe
            flow_value = "|".join([
                str(v["name"]).strip()
                for v in json_parsed
            ])

            # Get the last value as adm_value for MIS lookup
            if json_parsed:
                adm_value = json_parsed[-1].get("name")
                if adm_value:
                    adm_value = adm_value.lower().strip()
                    adm_level = len(json_parsed) + 1  # +1 for national level

                    # Use pre-loaded lookup with substring matching
                    mis_value = self.find_mis_administration_match(
                        adm_value, adm_level
                    )

        except (json.JSONDecodeError, TypeError, KeyError, IndexError) as e:
            # Not a JSON cascade or malformed - keep original value
            logger.debug(f"Could not parse as cascade JSON: {e}")

        return flow_value, mis_value

    def split_administration_path(self, adm: str) -> Tuple[str, Optional[str]]:
        """
        Split an administration path into value and parent.

        Args:
            adm: Administration path (e.g., "Country|Province|District")

        Returns:
            Tuple of (value, parent) where parent is None for top-level
            entries
        """
        parts = adm.split("|")
        value = parts[-1].lower().strip()
        parent = parts[-2].strip() if len(parts) > 1 else None
        return value, parent

    def find_flow_administration_match(
        self,
        flow_adm_lookup: Dict[Tuple[str, Optional[str]], Dict[str, Any]],
        adm_value: str,
        adm_parent: Optional[str]
    ) -> Optional[Dict[str, Any]]:
        """
        Find a matching Flow administration using exact or substring match.

        Args:
            flow_adm_lookup: Pre-loaded Flow administration lookup dictionary
            adm_value: The administration name to match (lowercase, trimmed)
            adm_parent: The parent administration name (or None)

        Returns:
            The matching administration entry, or None if not found
        """
        # First try exact match
        lookup_key = (adm_value, adm_parent)
        if lookup_key in flow_adm_lookup:
            return flow_adm_lookup[lookup_key]

        # If no exact match, try substring matching
        # This handles cases like "Nadroga" matching "Nadroga/Navosa"
        for key, entry in flow_adm_lookup.items():
            db_name, db_parent = key
            # Check if names match (case-insensitive, trimmed)
            if (adm_value in db_name or db_name in adm_value) and (
                (adm_parent is None and db_parent is None) or
                (adm_parent and db_parent and
                 (adm_parent in db_parent or db_parent in adm_parent))
            ):
                logger.debug(
                    f"Substring match: '{adm_value}' matched to "
                    f"'{db_name}' (ID: {entry['id']})"
                )
                return entry

        return None

    def process_survey_data(
        self,
        flow_cascades_df: pd.DataFrame,
        survey_id: int,
        survey_rows: pd.DataFrame
    ) -> Tuple[List[Dict[str, Any]], Dict[str, Set[str]]]:
        """
        Process data for a single survey.

        Args:
            flow_cascades_df: The combined cascade DataFrame
            survey_id: The survey ID to process
            survey_rows: Rows for this survey from flow_cascades_df

        Returns:
            Tuple of (flow_adms_list, missing_adms_dict)
        """
        flow_adms_list = []
        missing_adms_dict: Dict[str, Set[str]] = {}

        # Find data file for this survey
        pattern = f"{survey_id}_*.csv"
        data_files = list(self.config.flow_data_path.glob(pattern))

        if not data_files:
            logger.warning(f"No data file found for survey_id {survey_id}")
            return flow_adms_list, missing_adms_dict

        data_file_path = data_files[0]
        logger.info(
            f"Processing survey {survey_id} from {data_file_path.name}"
        )

        try:
            data_df = pd.read_csv(data_file_path)
        except Exception as e:
            logger.error(f"Failed to read data file {data_file_path}: {e}")
            return flow_adms_list, missing_adms_dict

        # Track seen entries to avoid duplicates
        seen_entries: Set[Tuple[int, int, int]] = set()

        for _, row in survey_rows.iterrows():
            question_id = row['question_id']
            sqlite_file = row['cascade_resource']

            # Check if question_id column exists
            question_col = str(question_id)
            if question_col not in data_df.columns:
                logger.warning(
                    f"Question ID {question_id} not found in data for "
                    f"survey {survey_id}"
                )
                continue

            # Initialize missing_adms for this sqlite_file
            if sqlite_file not in missing_adms_dict:
                missing_adms_dict[sqlite_file] = set()

            # Process all answers for this question
            answers = data_df[question_col]

            for idx, answer_value in answers.items():
                if pd.isna(answer_value):
                    continue

                # Parse the cascade answer
                flow_value, mis_value = self.parse_cascade_answer(
                    str(answer_value)
                )

                # Track missing administrations
                if mis_value is None and flow_value:
                    missing_adms_dict[sqlite_file].add(flow_value)

                # Get datapoint_id for deduplication
                if 'datapoint_id' in data_df.columns:
                    datapoint_id = data_df.loc[idx, 'datapoint_id']
                else:
                    datapoint_id = idx

                # Create unique key for deduplication
                entry_key = (datapoint_id, survey_id, question_id)

                if entry_key not in seen_entries:
                    seen_entries.add(entry_key)
                    flow_adms_list.append({
                        "flow_datapoint_id": datapoint_id,
                        "flow_form_id": survey_id,
                        "flow_question_id": question_id,
                        "flow_value": flow_value,
                        "mis_value": mis_value,
                    })

        return flow_adms_list, missing_adms_dict

    def process_missing_administrations(
        self, missing_adms: Dict[str, Set[str]]
    ) -> List[Dict[str, Any]]:
        """
        Process missing administrations and find them in Flow SQLite files.

        Args:
            missing_adms: Dictionary of sqlite_file -> set of missing admin
                values

        Returns:
            List of dictionaries with administration data
        """
        flow_missing_adms_list = []

        for sqlite_file, adm_set in missing_adms.items():
            logger.info(
                f"Processing {len(adm_set)} missing administrations from "
                f"{sqlite_file}"
            )

            # Load flow administration data for this sqlite file
            flow_adm_lookup = self.load_flow_administration_data(sqlite_file)

            if not flow_adm_lookup:
                continue

            for adm in adm_set:
                adm_value, adm_parent = self.split_administration_path(adm)

                # Use pre-loaded lookup with substring matching
                adm_match = self.find_flow_administration_match(
                    flow_adm_lookup, adm_value, adm_parent
                )

                if adm_match:
                    flow_missing_adms_list.append({
                        "id": adm_match['id'],
                        "code": adm_match['code'],
                        "name": adm_match['name'],
                        "full_path": adm,
                    })
                else:
                    logger.debug(f"Could not find administration: {adm}")

        return flow_missing_adms_list

    def run(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Main execution method to process all cascade data.

        Returns:
            Tuple of (flow_adms_df, flow_missing_adms_df)
        """
        # Validate paths
        self.validate_paths()

        # Load cascade data
        flow_cascades_df = self.load_flow_cascades()

        if flow_cascades_df.empty:
            logger.warning("No cascade data to process")
            empty_df1 = pd.DataFrame(
                columns=self.config.flow_adms_columns
            )
            empty_df2 = pd.DataFrame(
                columns=self.config.flow_missing_adms_columns
            )
            return empty_df1, empty_df2

        # Check required columns
        if "cascade_resource" not in flow_cascades_df.columns:
            logger.error(
                f"Column 'cascade_resource' not found. Available columns: "
                f"{list(flow_cascades_df.columns)}"
            )
            empty_df1 = pd.DataFrame(
                columns=self.config.flow_adms_columns
            )
            empty_df2 = pd.DataFrame(
                columns=self.config.flow_missing_adms_columns
            )
            return empty_df1, empty_df2

        # Pre-load MIS administration data
        self.load_mis_administration_data()

        # Group by survey_id and process each survey
        survey_grouped = flow_cascades_df.groupby("survey_id")
        logger.info(f"Processing {len(survey_grouped)} surveys")

        all_flow_adms_list = []
        all_missing_adms: Dict[str, Set[str]] = {}

        for survey_id, survey_rows in survey_grouped:
            flow_adms_list, missing_adms = self.process_survey_data(
                flow_cascades_df,
                survey_id,
                survey_rows
            )

            all_flow_adms_list.extend(flow_adms_list)

            # Merge missing administrations
            for sqlite_file, adm_set in missing_adms.items():
                if sqlite_file not in all_missing_adms:
                    all_missing_adms[sqlite_file] = adm_set
                else:
                    all_missing_adms[sqlite_file].update(adm_set)

        # Create flow_adms DataFrame
        if all_flow_adms_list:
            flow_adms_df = pd.DataFrame(all_flow_adms_list)
        else:
            flow_adms_df = pd.DataFrame(
                columns=self.config.flow_adms_columns
            )

        # Convert mis_value to nullable integer type
        if not flow_adms_df.empty and 'mis_value' in flow_adms_df.columns:
            try:
                flow_adms_df['mis_value'] = (
                    flow_adms_df['mis_value'].astype('Int64')
                )
            except Exception as e:
                logger.warning(f"Could not convert mis_value to Int64: {e}")

        logger.info(
            f"Created flow_adms DataFrame: {len(flow_adms_df)} rows"
        )

        # Process missing administrations
        flow_missing_adms_list = (
            self.process_missing_administrations(all_missing_adms)
        )

        # Create flow_missing_adms DataFrame
        if flow_missing_adms_list:
            flow_missing_adms_df = pd.DataFrame(flow_missing_adms_list)
        else:
            flow_missing_adms_df = pd.DataFrame(
                columns=self.config.flow_missing_adms_columns
            )

        logger.info(
            f"Created flow_missing_adms DataFrame: "
            f"{len(flow_missing_adms_df)} rows"
        )

        return flow_adms_df, flow_missing_adms_df

    def save_results(
        self, flow_adms_df: pd.DataFrame,
        flow_missing_adms_df: pd.DataFrame
    ) -> None:
        """Save the resulting DataFrames to CSV files."""
        # Ensure output directory exists
        self.config.output_mapping_csv.parent.mkdir(
            parents=True, exist_ok=True
        )
        self.config.output_missing_csv.parent.mkdir(
            parents=True, exist_ok=True
        )

        # Save flow_adms
        flow_adms_df.to_csv(self.config.output_mapping_csv, index=False)
        logger.info(
            f"Flow administration mapping saved to "
            f"{self.config.output_mapping_csv}"
        )

        # Save flow_missing_adms
        flow_missing_adms_df.to_csv(
            self.config.output_missing_csv, index=False
        )
        logger.info(
            f"Flow missing administration saved to "
            f"{self.config.output_missing_csv}"
        )


In [5]:
try:
    # Initialize configuration
    config = Configuration(base_path=".")

    # Create mapper instance
    mapper = AdministrationMapper(config)

    # Run the mapping process
    flow_adms_df, flow_missing_adms_df = mapper.run()

    # Save results
    mapper.save_results(flow_adms_df, flow_missing_adms_df)

    # Print summary
    print("\n" + "=" * 60)
    print("PROCESSING COMPLETE")
    print("=" * 60)
    print(f"Total administration mappings: {len(flow_adms_df)}")
    print(f"Total missing administrations: {len(flow_missing_adms_df)}")
    print("Output files:")
    print(f"  - {config.output_mapping_csv}")
    print(f"  - {config.output_missing_csv}")
    print("=" * 60 + "\n")
except Exception as e:
    logger.error(f"Script failed: {e}", exc_info=True)

2026-01-16 08:05:29,078 - INFO - Loading cascade files from output/flow_cascades
2026-01-16 08:05:29,079 - INFO - Found 3 cascade CSV files
2026-01-16 08:05:29,082 - INFO - Combined cascade data: 3 rows
2026-01-16 08:05:29,082 - INFO - Pre-loading MIS administration data...
2026-01-16 08:05:29,099 - INFO - Loaded 109 MIS administration entries
2026-01-16 08:05:29,101 - INFO - Processing 3 surveys
2026-01-16 08:05:29,103 - INFO - Processing survey 1520924 from 1520924_EPS_Water_Quality_Testing.csv
2026-01-16 08:05:29,143 - INFO - Processing survey 2490944 from 2490944_Rural_Water_Project_Inspection.csv
2026-01-16 08:05:29,158 - INFO - Processing survey 5530933 from 5530933_EPS_Projects_Construction_Monitoring.csv
2026-01-16 08:05:29,164 - INFO - Created flow_adms DataFrame: 777 rows
2026-01-16 08:05:29,165 - INFO - Processing 67 missing administrations from cascade-5430921-v6.sqlite
2026-01-16 08:05:29,165 - INFO - Pre-loading flow administration data from cascade-5430921-v6.sqlite...
2


PROCESSING COMPLETE
Total administration mappings: 777
Total missing administrations: 67
Output files:
  - ../../storage/akvo-flow/akvo-flow/administration_mapping.csv
  - ../../storage/akvo-flow/akvo-flow/administration_missing.csv

